# Registered 3PTB searches and saved-pose evaluation

Scientific question: does the declared dry trypsin / benzamidinium +1 hypothesis generate near-reference poses and rank one first? The [prospective protocol](3ptb_prospective_protocol.md) fixes two torsion arms, two 20 Å boxes, seeds 7/42/2026, efforts 1/8, CPU 1 and Vina 1.2.7. Near-reference means positional RMSD of the original nine heavy identities ≤2 Å, with no alignment or symmetry matching.

All 24 searches returned 167 poses. The original producer failed to evaluate them because its case-specific selection wrapper indexed an absent optional map. Its original results, errors and producer remain immutable. The separate receiving evaluation corrects that wrapper and supplies the registered measurements without repeating docking or scoring. A receiving constructor check also rejected duplicate score descriptors before any RMSD; the corrected receiver supplies them once.

This human Python/Jupyter workflow uses public molecular APIs and the explicit case orchestration. It follows the pilots' general human-workflow principle and uses only public 3PTB data. Molecular operations and geometry stay with MolSysMT. Preparation and source handoff are documented in the [admission notebook](3ptb_admission_2026-10-10.ipynb); compressed input remains tracked in MolSysMT #385.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gzip, json, subprocess, sys
import numpy as np
import pandas as pd
import molsysmt as msm
import pyunitwizard as puw
import dockingmt as dmt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'devtools/qualify_3ptb_search.py').is_file())
sys.path.insert(0, str(root))
from devtools.qualify_3ptb_admission import DATA, admit, sha
from devtools.qualify_3ptb_search import OUTPUT, baseline, evaluate, execution_protocol, verify_replay
from devtools.evaluate_3ptb_saved import OUTPUT as EVALUATIONS

checkpoint = json.loads(OUTPUT.with_name('checkpoint_2026-10-10.json').read_text())
assert sha(OUTPUT.read_bytes()) == checkpoint['original_producer']['archive_sha256']
assert sha(EVALUATIONS.read_bytes()) == checkpoint['receiving_evaluation']['archive_sha256']
searches = json.loads(gzip.decompress(OUTPUT.read_bytes()))
received = json.loads(gzip.decompress(EVALUATIONS.read_bytes()))
original, registration = baseline()
assert searches['execution_registration'] == registration
assert sha(OUTPUT.with_name('producer_2026-10-10.py.txt').read_bytes()) == searches['original_producer']['consumer_driver_sha256']
assert received['search_archive_sha256'] == sha(OUTPUT.read_bytes())
print('Original search summary, including reporting failures:')
display(pd.DataFrame(searches['summary']))
print('Separate receiving evaluation:')
display(pd.DataFrame(received['summary']))

Original search summary, including reporting failures:


,any_near,arm,attempted,domain,evaluated,evaluation_failures,planned,returned,returned_poses,search_failures,top1_near
0,0,rigid,6,native,0,6,6,6,41,0,0
1,0,rigid,6,negative,0,6,6,6,39,0,0
2,0,flexible,6,native,0,6,6,6,41,0,0
3,0,flexible,6,negative,0,6,6,6,46,0,0


Separate receiving evaluation:


,any_near,arm,attempted,domain,evaluated,evaluation_failures,planned,returned,returned_poses,search_failures,top1_near
0,6,rigid,6,native,6,0,6,6,41,0,6
1,0,rigid,6,negative,6,0,6,6,39,0,0
2,6,flexible,6,native,6,0,6,6,41,0,6
3,0,flexible,6,negative,6,0,6,6,46,0,0


## Inspect and replay the admitted molecular hypothesis

Use the qualified source/runtime composition listed in the [admission record](3ptb_admission.md). The explicit fixed states, disulfides, H, Gasteiger–Marsili charges and AutoDock4 types remain unassessed as a scientific model. Preparation is replayed once here, without any new native maps, scoring, optimization or search. Literal PDBQT bytes and frozen boxes/population must match the prior admission before they could be reused.

The initial execution handoff records constructor `None`, whose resolved protocol serialization is `[]`. This removes an automatic-preparation request from the executable call; the already prepared flexible PDBQT keeps its C1–C branch. Original and executable declarations remain separate.

In [2]:
with TemporaryDirectory(prefix='dockingmt-3ptb-search-notebook-source-') as directory:
    plain = Path(directory)/'3PTB.pdb'
    with plain.open('wb') as output:
        subprocess.run(['gzip','-dc',str(DATA/'3PTB.pdb.gz')], stdout=output, check=True)
    receptor, partners, reference, boxes, replay = admit(plain)
verify_replay(original, replay)
assert len(original['population']) == 24
print('Exact admitted input bytes, coordinates, boxes and population reproduced.')
print('Written → original full-source maps:', {arm: original['prepared_inputs'][arm]['pdbqt_to_full_source_atom_indices'] for arm in ('rigid','flexible')})
print('Reference names:', msm.get(reference, element='atom', atom_name=True))
print('Original backend default signature:', searches['native_dock_signature'])

Exact admitted input bytes, coordinates, boxes and population reproduced.
Written → original full-source maps: {'rigid': [1630, 1631, 1632, 1633, 1634, 1635, 1636, 1637, 1638, None, None, None, None], 'flexible': [1636, 1637, 1638, None, None, None, None, 1630, 1631, 1632, 1633, 1634, 1635]}
Reference names: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C', 'N1', 'N2']
Original backend default signature: (self, exhaustiveness=8, n_poses=20, min_rmsd=1.0, max_evals=0)


## Verify every saved returned pose under another unit policy

The helper selects the admitted nine observed identities, verifies original IDs/names/elements/residue identities, builds a heavy-only pose result, and calls public `dockingmt.evaluate_redocking`. Its positional RMSD is supplied by MolSysMT. The full thirteen-atom result and all five native energy components stay untouched. No general atom matcher or molecular geometry implementation is introduced downstream.

In [3]:
pose_rows = []
run_rows = []
with puw.context(standard_units=['pm','fs','K','mole','dalton','e','kJ/mol','degrees']):
    for raw, measured in zip(searches['runs'], received['runs'], strict=True):
        result = dmt.DockingResult.from_dict(raw['result'])
        dmt.verify_captured_inputs(result.provenance['backend_artifacts'])
        assert result.protocol_info == execution_protocol(raw).to_dict()
        for role in ('receptor','partner'):
            assert result.provenance['backend_artifacts'][role]['sha256'] == raw[f'{role}_pdbqt_sha256']
        actual_box = result.provenance['backend_box']
        assert {key: actual_box[key] for key in raw['backend_box']} == raw['backend_box']
        verified = evaluate(result, reference, boxes[raw['domain']], original, raw['arm'])
        np.testing.assert_allclose([p['rmsd'] for p in verified['report']['poses']], [p['rmsd'] for p in measured['evaluation']['report']['poses']], rtol=0, atol=1e-10)
        assert verified['containment'] == measured['evaluation']['containment']
        assert measured['original_status'] == raw['status'] == 'evaluation_failed'
        for pose, vector, containment in zip(verified['report']['poses'], verified['complete_energy_vectors'], verified['containment'], strict=True):
            pose_rows.append({'run':raw['index'], 'arm':raw['arm'], 'domain':raw['domain'], 'rank':pose['rank'], 'rmsd_A':pose['rmsd'], 'near':pose['recovered'], 'five_native_components_kcal_mol':vector, 'all_heavy_inside':all(containment['heavy']), 'all_full_inside':all(containment['full'])})
        run_rows.append({'index':raw['index'], 'arm':raw['arm'], 'domain':raw['domain'], 'seed':raw['protocol']['parameters']['seed'], 'effort':raw['protocol']['parameters']['exhaustiveness'], 'poses':len(result.poses), 'top1_rmsd_A':verified['report']['first_pose']['rmsd'], 'best_rmsd_A':verified['best_rmsd_angstrom'], 'first_near_rank':verified['first_near_rank'], 'top1_near':verified['top1_near'], 'any_near':verified['any_near']})
poses = pd.DataFrame(pose_rows)
runs = pd.DataFrame(run_rows)
assert len(poses) == 167 and len(runs) == 24
print('All 167 saved poses verified in explicit angstrom under pm/fs/degree/e standards.')
display(runs)

All 167 saved poses verified in explicit angstrom under pm/fs/degree/e standards.


,index,arm,domain,seed,effort,poses,top1_rmsd_A,best_rmsd_A,first_near_rank,top1_near,any_near
0,0,rigid,native,7,1,5,1.933697,0.228346,1.0,True,True
1,1,rigid,native,7,8,9,1.938350,0.320282,1.0,True,True
2,2,rigid,native,42,1,4,1.934704,1.934704,1.0,True,True
3,3,rigid,native,42,8,9,1.929823,1.929823,1.0,True,True
4,4,rigid,native,2026,1,5,1.933540,1.933540,1.0,True,True
5,5,rigid,native,2026,8,9,1.944668,0.264557,1.0,True,True
6,6,rigid,negative,7,1,4,65.312391,62.073958,NaN,False,False
7,7,rigid,negative,7,8,9,65.312391,57.461948,NaN,False,False
8,8,rigid,negative,42,1,4,55.524896,55.457340,NaN,False,False
9,9,rigid,negative,42,8,9,55.524896,55.070225,NaN,False,False


In [4]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(poses)
outside = poses.loc[~poses.all_full_inside, ['run','arm','domain','rank','rmsd_A','all_heavy_inside']]
print('Seven returned poses have at least one retained atom outside the literal box:')
display(outside)
assert len(outside) == 7
assert (~poses.all_heavy_inside).sum() == 3
assert poses.loc[(poses.domain=='native') & (poses['rank']==1), 'all_full_inside'].all()
assert not poses.loc[poses.domain=='negative', 'near'].any()
print('Containment is observed per pose; no returned pose is discarded.')

,run,arm,domain,rank,rmsd_A,near,five_native_components_kcal_mol,all_heavy_inside,all_full_inside
0,0,rigid,native,1,1.933697,True,"[-6.422, -6.422, 0.0, 0.0, 0.0]",True,True
1,0,rigid,native,2,0.228346,True,"[-6.258, -6.258, 0.0, 0.0, 0.0]",True,True
2,0,rigid,native,3,9.998747,False,"[-4.104, -4.104, 0.0, 0.0, 0.0]",True,True
3,0,rigid,native,4,9.935509,False,"[-4.006, -4.006, 0.0, 0.0, 0.0]",True,True
4,0,rigid,native,5,11.298291,False,"[-3.507, -3.507, 0.0, 0.0, 0.0]",False,False
5,1,rigid,native,1,1.938350,True,"[-6.43, -6.43, 0.0, 0.0, 0.0]",True,True
6,1,rigid,native,2,0.320282,True,"[-6.342, -6.342, 0.0, 0.0, 0.0]",True,True
7,1,rigid,native,3,3.502848,False,"[-5.271, -5.271, 0.0, 0.0, 0.0]",True,True
8,1,rigid,native,4,3.963112,False,"[-5.172, -5.172, 0.0, 0.0, 0.0]",True,True
9,1,rigid,native,5,3.478740,False,"[-5.117, -5.117, 0.0, 0.0, 0.0]",True,True


Seven returned poses have at least one retained atom outside the literal box:


,run,arm,domain,rank,rmsd_A,all_heavy_inside
4,0,rigid,native,5,11.298291,False
77,11,rigid,negative,7,56.442415,True
103,15,flexible,native,5,10.990081,False
119,17,flexible,native,8,11.018946,False
132,19,flexible,negative,5,63.219339,True
139,20,flexible,negative,3,57.082527,True
148,21,flexible,negative,4,57.082527,True


Containment is observed per pose; no returned pose is discarded.


## Human execution entry point and interpretation

The complete public-API composition is inspectable in `devtools/qualify_3ptb_search.py`: prepare once, authenticate the exact prior admission, project only the preparation-intent argument, construct `DockingProblem`, call `dockingmt.dock`, preserve every attempt/result, and evaluate each retained heavy identity through the public evaluator. To conduct a separate replication, restore the unchanged PDB with `gzip -dc` and run `python -m devtools.qualify_3ptb_search 3PTB.pdb --output /tmp/new-3ptb-replication.json.gz`. Use a new path and preserve its new producer identity. This notebook defaults to replaying evidence and does not repeat the registered searches.

Both native-box arms have near-reference rank 1 in all six planned rows; the displaced-domain control has none among any returned poses. Most first-pose source-identity RMSDs are close to the declared 2 Å boundary, and the best returned RMSD may be lower. No post-outcome symmetry normalization or threshold adjustment is applied. ROOT orientation differs between arms, so this finite result does not isolate torsional flexibility as a mechanism.

Seven poses extend partly outside the literal box (three include heavy atoms); all native first poses are inside. These remain in the full results and denominator. The negative box is a methodological domain control, not a nonbinder. This single hypothesis/case establishes neither experimental affinity, state/model suitability, recovery probabilities nor multi-complex generalization. Symmetry-aware molecular comparison remains a separately registered future provider workflow under MolSysMT #310; environmental H and preparation/projection questions remain with their existing owners. No PharmacophoreMT operation is needed.